# Blazar analysis with *fermipy*: single-source walkthrough
Adapted from HEARG Tutorial 2 (Fermi tools) and Tutorial 3 (fermipy). Same chain of steps, but the target is a **blazar** and we add what a blazar paper needs: a **PowerLaw vs LogParabola curvature test**, a **monthly light curve** and **variability statistics** ($F_{\rm var}$, flare duty cycle).

Run this notebook for **one** source first (e.g. `3C 279`) to check your setup, then use `run_fermipy.py` for the whole sample.

Before starting: `python verify_sources.py`, and put the LAT photon (`*_PH*.fits`) and spacecraft (`*_SC*.fits`) files in `data/<slug>/`.

In [ ]:
import sys, json
sys.path.insert(0, '..')          # so we can import blazarlib / run_fermipy from the project root
from pathlib import Path
import numpy as np, yaml, matplotlib.pyplot as plt
import blazarlib as bl
import run_fermipy as rf

SOURCE = '3C 279'                 # <- change me (must be a name in sources.csv)
s = bl.load_settings('../settings.yaml')

## 1. Resolve the source in the catalog and build the config
Unlike Tutorial 3, `config.yaml` is generated from `settings.yaml`, so every source in the sample is analysed identically.

In [ ]:
cat = bl.Catalog(s['paths']['catalog'])
srcs = bl.read_sources(Path(s['_root']) / s['paths']['sources'])
r = srcs[srcs['name'] == SOURCE].iloc[0]
hit = cat.lookup(r['fgl_name'], r.get('assoc'))
row = {**r.to_dict(), **hit}; row['name'], row['class'] = r['name'], r['class']
print(hit)

slug = bl.slugify(SOURCE)
outdir = Path(s['_root']) / s['paths']['results_dir'] / slug
outdir.mkdir(parents=True, exist_ok=True)
evfile, scfile = bl.find_data(slug, s)
cfg = bl.build_fermipy_config(row, s, outdir, evfile, scfile)
(outdir / 'config.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))

## 2. Set up the analysis (runs gtselect, gtmktime, gtltcube, gtbin, gtexpcube2, gtsrcmaps)
> This is the slow step (minutes to tens of minutes for a multi-year dataset).

In [ ]:
from fermipy.gtanalysis import GTAnalysis
gta = GTAnalysis(str(outdir / 'config.yaml'), logging={'verbosity': 3})
gta.setup()
gta.optimize()
gta.print_roi()

## 3. Free parameters and fit
Same logic as Tutorial 3: free the normalisations of nearby sources and the diffuse models, and let the *target's* spectral shape float (blazars are bright enough to constrain it).

In [ ]:
name = row['fgl_name']
a = s['analysis']
gta.free_sources(distance=a['free_radius'], pars='norm')
gta.free_source('galdiff', pars='norm')
gta.free_source('isodiff', pars='norm')
gta.free_source(name)
fit = gta.fit(optimizer=a['optimizer'])
print('fit success:', fit.get('fit_success'))
gta.print_params()

## 4. Check the model: residual map and TS map
The residual map should look like noise (|significance| < ~3). Structured residuals mean a missing source or bad diffuse model - fix this before trusting any spectrum or light curve.

In [ ]:
gta.residmap(make_plots=True)
gta.tsmap(make_plots=True)
# diagnostic plots are written to the output directory:
print(sorted(p.name for p in outdir.glob('*residmap*.png'))[:5])

## 5. Curvature test: PowerLaw vs LogParabola
$TS_{\rm curv} = 2(\ln\mathcal{L}_{\rm LP} - \ln\mathcal{L}_{\rm PL})$. Values above about 4 prefer a curved spectrum. FSRQs usually show strong curvature (a spectral break near a few GeV); many BL Lacs are closer to a power law.

In [ ]:
curv = rf.curvature_test(gta, name, a['curvature_ts_threshold'], a['optimizer'])
print(curv)
summary = rf.summarize(gta, name, row); summary.update(curv)
print({k: summary[k] for k in ('ts', 'flux', 'index', 'index_err', 'beta', 'ts_curv', 'model')})

## 6. Localisation and SED
A blazar should sit on its radio counterpart: check `loc_offset` against `loc_r95`.

In [ ]:
loc = gta.localize(name, update=False, make_plots=False)
print('best-fit RA, Dec:', loc['ra'], loc['dec'], ' offset [deg]:', loc['pos_offset'], ' r95 [deg]:', loc['pos_r95'])
sed = gta.sed(name, make_plots=False, outfile=f'{slug}_sed.fits')
np.savez(outdir / 'sed.npz', **rf.arrays_only(sed))

In [ ]:
e, lo, hi = sed['e_ref'], sed['e_ref'] - sed['e_min'], sed['e_max'] - sed['e_ref']
det = sed['ts'] >= 4
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.errorbar(e[det], sed['e2dnde'][det], xerr=[lo[det], hi[det]], yerr=sed['e2dnde_err'][det], fmt='o', color='C3')
ax.errorbar(e[~det], sed['e2dnde_ul95'][~det], xerr=[lo[~det], hi[~det]], yerr=0.35*sed['e2dnde_ul95'][~det],
            uplims=True, fmt='none', color='C3', alpha=.6)
ax.set(xscale='log', yscale='log', xlabel='E [MeV]', ylabel=r'E$^2$dN/dE [MeV cm$^{-2}$ s$^{-1}$]', title=SOURCE)
plt.show()

## 7. Light curve and variability
Monthly bins by default (`lightcurve.bin_days` in `settings.yaml`). Bins with TS < 4 are shown as 95% upper limits and excluded from $F_{\rm var}$.

In [ ]:
L = s['lightcurve']
lc = gta.lightcurve(name, binsz=L['bin_days']*86400., free_radius=L['free_radius'], use_scaled_srcmap=True,
                    multithread=True, nthread=L['nthread'], save_bin_data=False, make_plots=False,
                    outfile=f'{slug}_lightcurve.fits')
np.savez(outdir / 'lightcurve.npz', **rf.arrays_only(lc))
st = bl.lightcurve_stats(lc, L['ts_min'], L['flare_factor']); summary.update(st)
print({k: st[k] for k in ('lc_nbins', 'lc_ndet', 'fvar', 'fvar_err', 'chi2_p', 'duty_cycle', 'peak_to_median')})

In [ ]:
k = 'flux' if 'flux' in lc else 'eflux'
tc = 0.5*(np.asarray(lc['tmin_mjd']) + np.asarray(lc['tmax_mjd']))
det = np.asarray(lc['ts']) >= L['ts_min']
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.errorbar(tc[det], np.asarray(lc[k])[det], yerr=np.asarray(lc[k+'_err'])[det], fmt='o', ms=4, color='C3')
ax.plot(tc[~det], np.asarray(lc[k+'_ul95'])[~det], 'v', color='C3', alpha=.5)
ax.axhline(L['flare_factor']*np.median(np.asarray(lc[k])[det]), ls='--', color='k', lw=.9)
ax.set(yscale='log', xlabel='MJD', ylabel='Flux [ph cm$^{-2}$ s$^{-1}$]', title=SOURCE)
plt.show()

## 8. Save
This writes the same `summary.json` that `run_fermipy.py` produces, so this source is picked up by the population analysis.

In [ ]:
summary['time'] = dict(start=s['time']['start'], end=s['time']['end'])
bl.dump_json(summary, outdir / 'summary.json')
gta.write_roi(f'{slug}_fit', make_plots=False)
print('saved', outdir / 'summary.json')